In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [8]:
df= pd.read_csv(r"C:\Users\ADMIN\Desktop\Protein_Expression_Yield_Prediction\Data\protein_expression.csv")
df.head(3)

,No,Protein,Signal_peptide,Signal_peptide_target_protein_sequence,Media,OD_au,IPTG_mM,Temperature_C,Time_h,Maximum_expression_yield_mg_L,Level,References
0,1,"Xylanase, xynA2",ompA,MKKTAIAIAVALAGFATVAQAMQTTPNSEG WHDGYYYSWWSDGGA...,LB,3,1,20,2,692.8,H,[1]
1,2,Human interleukin 6,phoA,MKQSTIALALLPLLFTPVTKAMPVPPGEDSKD VAAPHRQPLTSSE...,LB,0.5,1,22/37,4,0,L,[2]
2,3,Candida antarctica lipase B (CALB),pelB,MKYLLPTAAAGLLLLAAQPAMALPSGSDPA FSQPKSVLDAGLTCQ...,LB,0.5- 0.6,1,16,24,5.2,M,[3]


In [9]:
# Total no of proteins
total_no_of_protein = df['Protein'].count()
print(f"Total_no_of_protein: {total_no_of_protein}")

# UNique proteins
Uniques_proteins = df['Protein'].nunique()
print(f"Total_no_of_unique_protein: {Uniques_proteins}")


Total_no_of_protein: 97
Total_no_of_unique_protein: 80


In [10]:
# Shape
print(f"Shape of Dataset:{df.shape}")
print(f"columns: {df.columns}")

Shape of Dataset:(97, 12)
columns: Index(['No', 'Protein', 'Signal_peptide',
       'Signal_peptide_target_protein_sequence', 'Media', 'OD_au', 'IPTG_mM',
       'Temperature_C', 'Time_h', 'Maximum_expression_yield_mg_L', 'Level',
       'References'],
      dtype='str')


In [11]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicated values: {df.duplicated().sum().sum()}")

Missing values: 1
Duplicated values: 0


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 12 columns):
 #   Column                                  Non-Null Count  Dtype
---  ------                                  --------------  -----
 0   No                                      97 non-null     int64
 1   Protein                                 97 non-null     str  
 2   Signal_peptide                          97 non-null     str  
 3   Signal_peptide_target_protein_sequence  97 non-null     str  
 4   Media                                   96 non-null     str  
 5   OD_au                                   97 non-null     str  
 6   IPTG_mM                                 97 non-null     str  
 7   Temperature_C                           97 non-null     str  
 8   Time_h                                  97 non-null     str  
 9   Maximum_expression_yield_mg_L           97 non-null     str  
 10  Level                                   97 non-null     str  
 11  References                      

In [13]:
target = "Maximum_expression_yield_mg_L"

# Convert simple numeric values; ranges become NaN for now
df["Yield_clean"] = pd.to_numeric(df[target], errors="coerce")

print("Valid numeric yields:", df["Yield_clean"].notna().sum())
print("Values requiring review:", df["Yield_clean"].isna().sum())

display(df.loc[df["Yield_clean"].isna(), [target]].head(20))



Valid numeric yields: 78
Values requiring review: 19


,Maximum_expression_yield_mg_L
15,17.8- 45
40,0.8/ 0.3
43,Feb-15
55,0.028 – 0.227
62,15-84
80,0.01- 0.375
81,0.86- 1.56
83,98-234
84,0.13- 4.76
85,0.22- 4.69


In [14]:
df.columns

Index(['No', 'Protein', 'Signal_peptide',
       'Signal_peptide_target_protein_sequence', 'Media', 'OD_au', 'IPTG_mM',
       'Temperature_C', 'Time_h', 'Maximum_expression_yield_mg_L', 'Level',
       'References', 'Yield_clean'],
      dtype='str')

In [15]:
df=df.dropna(subset=["Yield_clean"])

print("Values requiring review:", df["Yield_clean"].isna().sum())


Values requiring review: 0


In [16]:
features =df [["Media","OD_au","IPTG_mM", "Temperature_C", "Time_h" ]]
features.head(3)

,Media,OD_au,IPTG_mM,Temperature_C,Time_h
0,LB,3,1,20,2
1,LB,0.5,1,22/37,4
2,LB,0.5- 0.6,1,16,24


In [17]:
features.info()

<class 'pandas.DataFrame'>
Index: 78 entries, 0 to 93
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Media          77 non-null     str  
 1   OD_au          78 non-null     str  
 2   IPTG_mM        78 non-null     str  
 3   Temperature_C  78 non-null     str  
 4   Time_h         78 non-null     str  
dtypes: str(5)
memory usage: 4.8 KB


In [18]:
# Sepearte feature and targetr
X= features
y= df['Yield_clean']

In [19]:
num_features = ["OD_au", "IPTG_mM", "Temperature_C", "Time_h"]
for col in num_features:
    features[col] = pd.to_numeric(features[col], errors="coerce")

cat_features = ["Media"]


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

In [20]:
X_train, X_test, y_train, y_test =train_test_split(X,y,test_size = 0.2,random_state=42)

In [21]:
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (62, 5)
Shape of X_test: (16, 5)
Shape of y_train: (62,)
Shape of y_test: (16,)


In [22]:
lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print("Model training completed!")

Model training completed!


In [23]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 3), "mg/L")
print("RMSE:", round(rmse, 3), "mg/L")
print("R² Score:", round(r2, 3))



MAE: 123.71 mg/L
RMSE: 221.684 mg/L
R² Score: 0.169
